## Experiment 1
The first experiment to reproduce is §5.1's first experiment, namely counting patterns as identified in random directions, neurons and random panels of sentences.

In [24]:
import json

panels = None
# load the annotations
with open('../../../panels/panels_dedup.json') as f:
    panels = json.load(f)

In [25]:
keys = panels.keys()
# get the keys/filenames for each category
cat = {
    'rand': [key for key in keys if 'rand' in key],
    'neur': [key for key in keys if 'neur' in key],
    'dir':  [key for key in keys if 'dir'  in key]
}

This is all the data loading needed.

In [26]:
# collect pattern counts here
data = {
    'rand': [],
    'neur': [],
    'dir':  []
}

# get pattern count for random panels
for key in cat['rand']:
    data['rand'].append(len(panels[key]))

In [27]:
# collect pattern count for directions
# initialize with 33 zeroes first
data['dir'] = [0 for _ in range(33)]

# then sum over all of them
for key in cat['dir']:
    # extract the direction index from the file name
    # our consistent naming scheme makes this okay
    idx = int(key.split('_')[2].split('.')[0])
    # add the pattern count to the tally
    data['dir'][idx] += len(panels[key])

# then divide by the amount of datasets
# we obtain an average per neuron as a result
for i in range(len(data['dir'])):
    data['dir'][i] /= 4.0

In [28]:
from collections import defaultdict

# collect pattern count for neurons
data['neur'] = defaultdict(int)

for key in cat['neur']:
    # same method of extracting the index as above
    idx = int(key.split('_')[2].split('.')[0])
    # add the pattern count
    data['neur'][idx] += len(panels[key])

# make it back into a list
data['neur'] = list(data['neur'].values())
# same way of obtaining the average
for i in range(len(data['neur'])):
    data['neur'][i] /= 4.0

Now that we have converted the data into an acceptable structure, we produce some graphs and statistics.

In [29]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json

# table text/label for each condition
clabel = { 'rand': 'Random sentences',
           'neur': 'BERT neurons', 
           'dir':  'Random directions' }

# table data frame
df = pd.DataFrame([{
    'Condition':   clabel[cname],
    'Panel count': len(cat[cname]),
    'μ':           np.array([len(panels[key]) for key in cat[cname]]).mean(),
    'σ':           np.array([len(panels[key]) for key in cat[cname]]).std(ddof=1),
    } 
    for cname in ['neur', 'dir', 'rand']
]).round(2)

df

,Condition,Panel count,μ,σ
0,BERT neurons,100,1.73,1.01
1,Random directions,132,1.45,0.81
2,Random sentences,29,0.21,0.49


In [30]:
# original paper % for comparison
paper_data = {
    'rand': 14,
    'neur': 80,
    'dir':  82
}

df = pd.DataFrame([{
    'Condition':           clabel[cname],
    'Panel count':         len(cat[cname]),
    'Has patterns (%)':    np.mean([len(panels[key]) > 0 for key in cat[cname]]) * 100,
    'Original result (%)': paper_data[cname],
    } 
    for cname in ['neur', 'dir', 'rand']
]).round(2)

df

,Condition,Panel count,Has patterns (%),Original result (%)
0,BERT neurons,100,87.00,80
1,Random directions,132,88.64,82
2,Random sentences,29,17.24,14
